# Running on Lambda or in the client

A query fans out whenever the planner in zarr-datafusion-lambda can do so.

Set `lambda.enabled` to `false` to make the whole query run locally in the client.

The following compares running on lambda with running in the client on a point query over the time-series layout, which reads only a chunk or two.

In [ ]:
import os

from zarr_datafusion_lambda import sql

# Where ERA5 and the workers are. Read once a kernel, at the first query.
os.environ["AWS_REGION"] = "us-east-1"

In [ ]:
point_tables = {
    "era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/temporal"}
}
point_query = """
SELECT date_trunc('month', CAST(valid_time AS TIMESTAMP)) AS month,
       avg(t2m)                                          AS mean_t2m
FROM era5
WHERE latitude = 51.5 AND longitude = 0
  AND valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2021-01-01'
GROUP BY 1
ORDER BY 1
"""
on_lambda = sql.Session(point_tables)
in_client = sql.Session(point_tables, config={"lambda.enabled": "false"})

Compare the query plan on lambda, vs running in the client. Lambda uses a `MapLambdaZarrExec` operation to fan out fragments to workers, whereas running in the client uses an in-process `ZarrExec` operation.

In [ ]:
print(on_lambda.explain(point_query))

In [ ]:
print(in_client.explain(point_query))

Now run both queries and compare their timings.

In [ ]:
%%time
on_lambda.sql(point_query, progress=True).to_pandas()

In [ ]:
%%time
in_client.sql(point_query).to_pandas()